# NEAT Fitness Tuning Notebook

Use this notebook to run short preset sweeps over the fitness weights in `miner_neat2.py`. The goal is fast behavior feedback, not final training quality.

The notebook keeps `miner_neat2.py` unchanged. It copies the relevant simulation loop locally so each run can return behavior metrics such as minerals collected, survival time, idle ratio, fuel use, and collision rate.

## 1. Paths and Display Mode

Set `HEADLESS = True` before running the import cell if you are on a machine without a display. Replay needs a real Pygame display, so it is skipped when headless.

In [1]:
from pathlib import Path
import os
import sys

PROJECT_DIR = Path.cwd()
if PROJECT_DIR.name != "AS4_space-miner":
    candidate = PROJECT_DIR / "AS4_space-miner"
    if candidate.exists():
        PROJECT_DIR = candidate

CONFIG_PATH = PROJECT_DIR / "neat_config.txt"
if not CONFIG_PATH.exists():
    raise FileNotFoundError(f"Could not find neat_config.txt at {CONFIG_PATH}")

# Change this before running the import cell if you are on a headless machine.
HEADLESS = False
if HEADLESS:
    os.environ["SDL_VIDEODRIVER"] = "dummy"
os.environ.setdefault("PYGAME_HIDE_SUPPORT_PROMPT", "1")

if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

print(f"Project directory: {PROJECT_DIR}")
print(f"NEAT config: {CONFIG_PATH}")

Project directory: c:\Users\Shira\NTUST\Soft-Computing\AS4_space-miner
NEAT config: c:\Users\Shira\NTUST\Soft-Computing\AS4_space-miner\neat_config.txt


## 2. Imports and Sweep Settings

In [2]:
import configparser
import math
import random
import tempfile
from statistics import mean

import neat
#import pygame

import miner_neat2 as miner

DEFAULT_FITNESS_WEIGHTS = miner.DEFAULT_FITNESS_WEIGHTS.copy()
WEIGHT_KEYS = list(DEFAULT_FITNESS_WEIGHTS)
EXPERIMENT_RESULTS = {}

print("Loaded miner_neat2.py")
print("Available fitness weights:", ", ".join(WEIGHT_KEYS))

ModuleNotFoundError: No module named 'neat'

In [ ]:
# Default short sweep settings.
GENERATIONS = 5
POP_SIZE = 40
MAX_TICKS = 800
EVAL_REPEATS = 2
FINAL_EVAL_SEEDS = 5
EXPERIMENT_SEED = 123

SETTINGS = {
    "GENERATIONS": GENERATIONS,
    "POP_SIZE": POP_SIZE,
    "MAX_TICKS": MAX_TICKS,
    "EVAL_REPEATS": EVAL_REPEATS,
    "FINAL_EVAL_SEEDS": FINAL_EVAL_SEEDS,
    "EXPERIMENT_SEED": EXPERIMENT_SEED,
}

# Quick smoke-test settings from the plan. Use these while checking mechanics.
SMOKE_SETTINGS = {
    **SETTINGS,
    "GENERATIONS": 1,
    "POP_SIZE": 6,
    "MAX_TICKS": 100,
    "EVAL_REPEATS": 1,
    "FINAL_EVAL_SEEDS": 2,
}

SETTINGS

## 3. Config and Table Helpers

In [ ]:
def load_base_weights(config_path=CONFIG_PATH):
    """Read the current [FitnessWeights] values from neat_config.txt."""
    parser = configparser.ConfigParser()
    read_files = parser.read(config_path)
    if not read_files:
        raise FileNotFoundError(f"Could not read {config_path}")

    weights = DEFAULT_FITNESS_WEIGHTS.copy()
    if parser.has_section("FitnessWeights"):
        for weight_name in WEIGHT_KEYS:
            if parser.has_option("FitnessWeights", weight_name):
                weights[weight_name] = parser.getfloat("FitnessWeights", weight_name)
    return weights


def make_temp_neat_config(weights, pop_size):
    """Create a temporary NEAT config with these fitness weights and pop size."""
    missing = sorted(set(WEIGHT_KEYS) - set(weights))
    if missing:
        raise ValueError(f"Missing fitness weights: {missing}")

    parser = configparser.ConfigParser()
    parser.read(CONFIG_PATH)

    if not parser.has_section("FitnessWeights"):
        parser.add_section("FitnessWeights")

    parser.set("NEAT", "pop_size", str(int(pop_size)))
    for weight_name in WEIGHT_KEYS:
        parser.set("FitnessWeights", weight_name, f"{float(weights[weight_name]):.12g}")

    temp_file = tempfile.NamedTemporaryFile(
        mode="w",
        suffix="_space_miner_neat_config.txt",
        delete=False,
        encoding="utf-8",
    )
    with temp_file:
        parser.write(temp_file)

    config = neat.Config(
        neat.DefaultGenome,
        neat.DefaultReproduction,
        neat.DefaultSpeciesSet,
        neat.DefaultStagnation,
        temp_file.name,
    )
    config.fitness_weights = dict(weights)
    return config, Path(temp_file.name)


NUMERIC_METRICS = [
    "fitness",
    "minerals",
    "alive_time",
    "mineral_progress",
    "idle_time",
    "idle_ratio",
    "fuel_used",
    "fuel_efficiency",
]


def summarize_trials(trials):
    if not trials:
        return {key: 0.0 for key in NUMERIC_METRICS}

    summary = {key: mean(float(trial[key]) for trial in trials) for key in NUMERIC_METRICS}
    summary["asteroid_collision_rate"] = mean(1.0 if trial["asteroid_collision"] else 0.0 for trial in trials)
    summary["out_of_fuel_rate"] = mean(1.0 if trial["out_of_fuel"] else 0.0 for trial in trials)
    return summary


def display_rows(rows):
    """Display rows as a DataFrame when pandas is present, otherwise print a compact table."""
    if not rows:
        print("No rows to display yet.")
        return rows

    try:
        import pandas as pd
    except ImportError:
        columns = list(rows[0])
        widths = {column: max(len(column), *(len(f"{row[column]}") for row in rows)) for column in columns}
        header = " | ".join(column.ljust(widths[column]) for column in columns)
        print(header)
        print("-" * len(header))
        for row in rows:
            print(" | ".join(f"{row[column]}".ljust(widths[column]) for column in columns))
        return rows

    return pd.DataFrame(rows)

## 4. Instrumented Trial Runner

This mirrors the movement, inputs, actions, mineral replenishment, asteroid movement, and fitness calculation in `miner_neat2.py`, but returns metrics for analysis.

In [ ]:
# def ensure_pygame_display(render=False):
#     if not pygame.get_init():
#         pygame.init()

#     if not render:
#         return

#     if HEADLESS:
#         raise RuntimeError("Replay needs a display. Set HEADLESS = False and rerun the import cells.")

#     if not pygame.display.get_init():
#         pygame.display.init()

#     miner.WIDTH, miner.HEIGHT = 800, 600
#     miner.screen = pygame.display.set_mode((miner.WIDTH, miner.HEIGHT))
#     pygame.display.set_caption("NEAT Fitness Tuning Replay")
#     miner.clock = pygame.time.Clock()


def run_trial(genome, config, weights, seed, max_ticks, render=False):
    """Run one seeded bot trial and return behavior metrics."""
    #ensure_pygame_display(render=render)

    random_state = random.getstate()
    random.seed(int(seed))
    try:
        net = neat.nn.FeedForwardNetwork.create(genome, config)
        ship = miner.Spaceship()
        minerals = [miner.Mineral() for _ in range(5)]
        asteroids = [miner.Asteroid() for _ in range(8)]
        alive_time = 0
        mineral_progress = 0.0
        mineral_best_distances = {}
        idle_time = 0
        asteroid_collision = False
        out_of_fuel = False
        no_minerals_left = False
        termination_reason = "max_ticks"

        while alive_time < max_ticks:
            alive_time += 1

            closest_mineral = min(
                minerals,
                key=lambda mineral: math.hypot(ship.x - mineral.x, ship.y - mineral.y),
                default=None,
            )
            if closest_mineral and closest_mineral not in mineral_best_distances:
                mineral_best_distances[closest_mineral] = math.hypot(
                    ship.x - closest_mineral.x,
                    ship.y - closest_mineral.y,
                )

            closest_asteroid = min(
                asteroids,
                key=lambda asteroid: math.hypot(ship.x - asteroid.x, ship.y - asteroid.y),
            )

            inputs = [
                math.hypot(ship.x - closest_mineral.x) / miner.WIDTH if closest_mineral else 0,
                math.atan2(closest_mineral.y - ship.y, closest_mineral.x - ship.x) / math.pi if closest_mineral else 0,
                math.hypot(ship.x - closest_asteroid.x) / miner.WIDTH,
                ship.fuel / 100.0,
            ]

            output = net.activate(inputs)

            movement_distance = 0.0
            ship.angle += (output[0] * 2 - 1) * 0.1
            if output[1] > 0.5:
                dx = ship.speed * math.cos(ship.angle)
                dy = ship.speed * math.sin(ship.angle)
                movement_distance = ship.move(dx, dy)

            if movement_distance == 0:
                idle_time += 1

            if closest_mineral:
                target_distance_after = math.hypot(ship.x - closest_mineral.x, ship.y - closest_mineral.y)
                best_distance = mineral_best_distances[closest_mineral]
                if target_distance_after < best_distance:
                    mineral_progress += best_distance - target_distance_after
                    mineral_best_distances[closest_mineral] = target_distance_after

            if output[2] > 0.5:
                ship.mine(minerals)
                if len(minerals) < 3:
                    minerals.extend(miner.Mineral() for _ in range(2))
                mineral_best_distances = {
                    mineral: distance
                    for mineral, distance in mineral_best_distances.items()
                    if mineral in minerals
                }

            for asteroid in asteroids:
                asteroid.move()

            asteroid_collision = any(
                math.hypot(ship.x - asteroid.x, ship.y - asteroid.y) < ship.radius + asteroid.radius
                for asteroid in asteroids
            )
            out_of_fuel = ship.fuel <= 0
            no_minerals_left = not minerals and ship.minerals == 0

            if render:
                miner.screen.fill(miner.BLACK)
                for mineral in minerals:
                    mineral.draw()
                for asteroid in asteroids:
                    asteroid.draw()
                ship.draw()
                pygame.display.flip()
                miner.clock.tick(30)

            if asteroid_collision:
                termination_reason = "asteroid_collision"
                break
            if out_of_fuel:
                termination_reason = "out_of_fuel"
                break
            if no_minerals_left:
                termination_reason = "no_minerals_left"
                break

        fuel_efficiency = ship.minerals / max(ship.fuel_used, 1)
        fitness = miner.calculate_fitness(
            ship,
            alive_time,
            mineral_progress,
            idle_time,
            asteroid_collision,
            weights,
        )
        genome.fitness = fitness

        return {
            "seed": int(seed),
            "fitness": float(fitness),
            "minerals": int(ship.minerals),
            "alive_time": int(alive_time),
            "mineral_progress": float(mineral_progress),
            "idle_time": int(idle_time),
            "idle_ratio": idle_time / max(alive_time, 1),
            "fuel_used": float(ship.fuel_used),
            "fuel_remaining": float(ship.fuel),
            "fuel_efficiency": float(fuel_efficiency),
            "asteroid_collision": bool(asteroid_collision),
            "out_of_fuel": bool(out_of_fuel),
            "no_minerals_left": bool(no_minerals_left),
            "termination_reason": termination_reason,
        }
    finally:
        random.setstate(random_state)

## 5. Short Training and Summary Helpers

In [ ]:
def run_short_training(preset_name, weights, settings):
    config, temp_config_path = make_temp_neat_config(weights, settings["POP_SIZE"])
    generation_history = []
    generation_counter = {"value": 0}
    seed_base = int(settings["EXPERIMENT_SEED"])

    # Reuse the same initial population for each preset, so differences mostly come from fitness weights.
    random.seed(seed_base)
    population = neat.Population(config)

    def eval_genomes(genomes, config):
        generation = generation_counter["value"]
        best_genome = None
        best_summary = None

        for genome_id, genome in genomes:
            trials = []
            for repeat in range(settings["EVAL_REPEATS"]):
                scenario_seed = seed_base + repeat * 1009
                trials.append(
                    run_trial(
                        genome,
                        config,
                        weights,
                        seed=scenario_seed,
                        max_ticks=settings["MAX_TICKS"],
                        render=False,
                    )
                )

            summary = summarize_trials(trials)
            genome.fitness = summary["fitness"]
            genome.tuning_metrics = summary

            if best_genome is None or genome.fitness > best_genome.fitness:
                best_genome = genome
                best_summary = summary

        generation_history.append(
            {
                "generation": generation + 1,
                "best_fitness": round(best_genome.fitness, 3),
                "best_minerals": round(best_summary["minerals"], 3),
                "best_alive_time": round(best_summary["alive_time"], 3),
                "best_idle_ratio": round(best_summary["idle_ratio"], 3),
                "best_collision_rate": round(best_summary["asteroid_collision_rate"], 3),
            }
        )
        print(
            f"{preset_name:18s} gen {generation + 1:02d}/{settings['GENERATIONS']:02d} "
            f"fitness={best_genome.fitness:8.2f} minerals={best_summary['minerals']:.2f} "
            f"idle={best_summary['idle_ratio']:.2f} collision={best_summary['asteroid_collision_rate']:.2f}"
        )
        generation_counter["value"] += 1

    winner = population.run(eval_genomes, settings["GENERATIONS"])
    training_fitness = float(winner.fitness)

    final_trials = []
    for index in range(settings["FINAL_EVAL_SEEDS"]):
        final_seed = seed_base + 50000 + index * 1009
        final_trials.append(
            run_trial(
                winner,
                config,
                weights,
                seed=final_seed,
                max_ticks=settings["MAX_TICKS"],
                render=False,
            )
        )

    result = {
        "preset_name": preset_name,
        "weights": dict(weights),
        "settings": dict(settings),
        "config": config,
        "temp_config_path": temp_config_path,
        "winner": winner,
        "training_fitness": training_fitness,
        "generation_history": generation_history,
        "final_trials": final_trials,
        "final_summary": summarize_trials(final_trials),
    }
    EXPERIMENT_RESULTS[preset_name] = result
    return result


def run_sweep(presets, settings=SETTINGS):
    results = {}
    for preset_name, weights in presets.items():
        print(f"\n=== {preset_name} ===")
        results[preset_name] = run_short_training(preset_name, weights, settings)
    return results


def make_summary_rows(results):
    rows = []
    for preset_name, result in results.items():
        summary = result["final_summary"]
        rows.append(
            {
                "preset": preset_name,
                "train_best_fitness": round(result["training_fitness"], 3),
                "eval_fitness": round(summary["fitness"], 3),
                "minerals": round(summary["minerals"], 3),
                "alive_time": round(summary["alive_time"], 3),
                "mineral_progress": round(summary["mineral_progress"], 3),
                "idle_ratio": round(summary["idle_ratio"], 3),
                "fuel_used": round(summary["fuel_used"], 3),
                "fuel_efficiency": round(summary["fuel_efficiency"], 3),
                "collision_rate": round(summary["asteroid_collision_rate"], 3),
                "out_of_fuel_rate": round(summary["out_of_fuel_rate"], 3),
            }
        )
    return sorted(rows, key=lambda row: row["eval_fitness"], reverse=True)


def make_history_rows(results):
    rows = []
    for preset_name, result in results.items():
        for row in result["generation_history"]:
            rows.append({"preset": preset_name, **row})
    return rows

## 6. Define Weight Presets

In [ ]:
BASE_WEIGHTS = load_base_weights()
assert BASE_WEIGHTS == miner.load_fitness_weights(str(CONFIG_PATH))


def preset(**updates):
    weights = BASE_WEIGHTS.copy()
    weights.update(updates)
    return weights


WEIGHT_PRESETS = {
    "baseline": BASE_WEIGHTS,
    "reward_mining": preset(minerals=BASE_WEIGHTS["minerals"] * 1.5),
    "pursue_minerals": preset(mineral_progress=BASE_WEIGHTS["mineral_progress"] * 5.0),
    "anti_idle": preset(idle_penalty=BASE_WEIGHTS["idle_penalty"] * 4.0),
    "fuel_saver": preset(fuel_efficiency=max(BASE_WEIGHTS["fuel_efficiency"] * 10.0, 0.1)),
    "collision_averse": preset(asteroid_collision_penalty=BASE_WEIGHTS["asteroid_collision_penalty"] * 10.0),
    "steering_averse": preset(steering_penalty=BASE_WEIGHTS["steering_penalty"] * 10.0),
}

weight_rows = [{"preset": name, **weights} for name, weights in WEIGHT_PRESETS.items()]
display_rows(weight_rows)

## 7. Run the Sweep

For a quick check, run only the smoke presets with `SMOKE_SETTINGS`. For the normal sweep, run all presets with `SETTINGS`.

In [ ]:
# Smoke test from the plan:
# RUN_PRESETS = {name: WEIGHT_PRESETS[name] for name in ["baseline", "reward_mining"]}
# SWEEP_RESULTS = run_sweep(RUN_PRESETS, settings=SMOKE_SETTINGS)

# Normal short sweep:
RUN_PRESETS = WEIGHT_PRESETS
SWEEP_RESULTS = run_sweep(RUN_PRESETS, settings=SETTINGS)

## 8. Compare Results

In [ ]:
summary_rows = make_summary_rows(SWEEP_RESULTS)
summary_table = display_rows(summary_rows)
summary_table

In [ ]:
history_rows = make_history_rows(SWEEP_RESULTS)
history_table = display_rows(history_rows)
history_table

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt


summary_df = pd.DataFrame(summary_rows).set_index("preset")
metric_columns = ["eval_fitness", "minerals", "alive_time", "idle_ratio", "collision_rate"]
axes = summary_df[metric_columns].plot(kind="bar", subplots=True, layout=(3, 2), figsize=(12, 10), legend=False)
for axis in axes.flatten():
    axis.tick_params(axis="x", labelrotation=35)
plt.tight_layout()
plt.show()

history_df = pd.DataFrame(history_rows)
if not history_df.empty:
    pivot = history_df.pivot(index="generation", columns="preset", values="best_fitness")
    pivot.plot(figsize=(10, 5), marker="o", title="Best training fitness by generation")
    plt.ylabel("fitness")
    plt.tight_layout()
    plt.show()

## 9. Replay a Winner

Change `SELECTED_PRESET` to replay a different winner. This does not retrain; it uses the winner already stored in `SWEEP_RESULTS`.

In [ ]:
# def replay_winner(preset_name, max_ticks=1200, seed=None):
#     if HEADLESS:
#         print("Replay skipped because HEADLESS is True.")
#         return None

#     if preset_name not in EXPERIMENT_RESULTS:
#         raise KeyError(f"No result named {preset_name!r}. Run the sweep first.")

#     result = EXPERIMENT_RESULTS[preset_name]
#     replay_seed = seed if seed is not None else result["settings"]["EXPERIMENT_SEED"] + 90000
#     metrics = run_trial(
#         result["winner"],
#         result["config"],
#         result["weights"],
#         seed=replay_seed,
#         max_ticks=max_ticks,
#         render=True,
#     )
#     print(metrics)
#     return metrics


# SELECTED_PRESET = "baseline"
# replay_winner(SELECTED_PRESET, max_ticks=1200)